# Dataset, DataLoader и mini-batch training

Цель: понять, как PyTorch хранит примеры, формирует batches и организует обучение по эпохам.

## 1. Разминка — ответить до запуска кода

1. Чем epoch отличается от iteration?
2. Если в dataset 10 объектов и `batch_size=4`, сколько batches будет в одной эпохе?
3. Какого размера будет последний batch?
4. Зачем перемешивать train-данные?
5. Нужно ли обычно перемешивать validation/test? Почему?

**Мои ответы:**

1. Iteration — обработка одного batch: forward pass, backward pass и обычно один шаг оптимизатора. Epoch — один полный проход по всему обучающему Dataset.
2. Три batch: $\lceil 10 / 4 \rceil = 3$.
3. Последний batch будет содержать 2 объекта, если `drop_last=False`.
4. Перемешивание не даёт модели постоянно видеть данные в одном потенциально смещённом порядке и помогает mini-batches лучше представлять весь train dataset. Это особенно важно, если объекты отсортированы по классу или target.
5. Обычно validation/test не перемешивают: параметры на них не обновляются, а порядок не влияет на агрегированные метрики. Стабильный порядок улучшает воспроизводимость и упрощает сопоставление предсказаний с исходными объектами.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## 2. Dataset

`Dataset` отвечает за количество объектов и получение одного объекта по индексу. `TensorDataset` объединяет тензоры с одинаковым первым измерением.

In [2]:
x = torch.arange(10, dtype=torch.float32).reshape(-1, 1)
y = 2 * x + 1

dataset = TensorDataset(x, y)

print(len(dataset))
print(dataset[3])
# выведи формы feature и target одного объекта
print(dataset[3][0].shape, dataset[3][1].shape)

10
(tensor([3.]), tensor([7.]))
torch.Size([1]) torch.Size([1])


**Наблюдение:** что именно возвращает `dataset[index]` и почему это tuple?

`dataset[index]` возвращает один sample как tuple `(feature, target)`. `TensorDataset` сохраняет структуру tuple, потому что может объединять произвольное количество тензоров с одинаковым первым измерением.

## 3. DataLoader и batches

`DataLoader` выбирает индексы, получает объекты из Dataset и объединяет их в batch.

In [3]:
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
)

print(len(loader))
# пройди по loader и для каждого batch выведи:
#       номер batch, значения batch_x, batch_x.shape и batch_y.shape
for batch_x, batch_y in loader:
    # dataset.tensors.index((batch_x, batch_y))
    print(batch_x, batch_x.shape, batch_y.shape)

3
tensor([[0.],
        [1.],
        [2.],
        [3.]]) torch.Size([4, 1]) torch.Size([4, 1])
tensor([[4.],
        [5.],
        [6.],
        [7.]]) torch.Size([4, 1]) torch.Size([4, 1])
tensor([[8.],
        [9.]]) torch.Size([2, 1]) torch.Size([2, 1])


**Наблюдение:** сколько batches получилось и почему последний отличается по форме?

Получилось 3 batch. Первые два имеют форму `(4, 1)`, а последний — `(2, 1)`, потому что 10 объектов не делятся на `batch_size=4` без остатка и `drop_last=False`.

## 4. Shuffle

Сравни порядок объектов в двух последовательных эпохах.

In [4]:
shuffled_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
)

for epoch in range(2):
    print(f"epoch={epoch}")
    # пройди по shuffled_loader и напечатай batch_x.flatten().tolist()
    for batch_x, batch_y in shuffled_loader:
        print(batch_x.flatten().tolist())

epoch=0
[4.0, 3.0, 2.0, 6.0]
[7.0, 5.0, 0.0, 8.0]
[1.0, 9.0]
epoch=1
[6.0, 9.0, 3.0, 2.0]
[0.0, 8.0, 4.0, 7.0]
[1.0, 5.0]


**Наблюдение:** меняется ли порядок? Означает ли `shuffle=True`, что сами пары `(x, y)` могут рассинхронизироваться?

Порядок объектов меняется между эпохами. Пары `(x, y)` не рассинхронизируются: DataLoader перемешивает индексы samples, а `dataset[index]` возвращает уже связанную пару feature и target.

## 5. Mini-batch training

Реализуй обучение модели `y = 2x + 1`. Один проход по всему loader — одна epoch; один batch — одна iteration и одно обновление параметров.

In [5]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)


model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

In [6]:
num_epochs = 100
global_step = 0

model.train()

for epoch in range(num_epochs):
    epoch_loss_sum = 0.0
    seen_objects = 0

    for batch_x, batch_y in shuffled_loader:
        optimizer.zero_grad()
        prediction = model(batch_x)
        loss = loss_fn(prediction, batch_y)
        loss.backward()
        optimizer.step()

        # Важно: считаем средний epoch loss с учётом размера batch.
        batch_size = batch_x.shape[0]
        epoch_loss_sum += loss.item() * batch_size
        seen_objects += batch_size
        global_step += 1

    epoch_loss = epoch_loss_sum / seen_objects

    if epoch % 10 == 0:
        print(
            f"epoch={epoch:3d}, step={global_step:3d}, "
            f"loss={epoch_loss:.6f}"
        )

epoch=  0, step=  3, loss=32.166932
epoch= 10, step= 33, loss=0.074745
epoch= 20, step= 63, loss=0.037134
epoch= 30, step= 93, loss=0.024851
epoch= 40, step=123, loss=0.009257
epoch= 50, step=153, loss=0.004364
epoch= 60, step=183, loss=0.002408
epoch= 70, step=213, loss=0.001637
epoch= 80, step=243, loss=0.000641
epoch= 90, step=273, loss=0.000276


In [7]:
print(model.linear.weight, model.linear.bias)
# Полученные параметры должны быть близки к истинным weight=2 и bias=1.

Parameter containing:
tensor([[2.0052]], requires_grad=True) Parameter containing:
tensor([0.9779], requires_grad=True)


## 6. Эксперименты

Для каждого эксперимента заново создай модель и optimizer, чтобы сравнение начиналось с одинаковой инициализации.

1. Сравни `batch_size=1`, `4` и `10`. Сколько optimizer steps происходит за epoch?
2. Установи `drop_last=True`. Сколько объектов модель увидит за epoch при `batch_size=4`?
3. Сравни `shuffle=False` и `shuffle=True`. На этом простом dataset различие может быть небольшим — объясни, где shuffle станет особенно важен.
4. Намеренно усредни batch losses как обычное среднее, не учитывая размер последнего batch. Совпадает ли результат с корректным средним по объектам? Почему?

## 7. Итог — ответы как на собеседовании

Ответь без кода:

1. Чем отличаются `Dataset` и `DataLoader`?
2. Чем отличаются sample, batch, iteration и epoch?
3. Почему последний batch может быть меньше остальных?
4. Что делает `drop_last=True` и когда это полезно?
5. Почему train loader обычно использует shuffle, а validation loader — нет?
6. Как корректно посчитать средний loss эпохи, если batches имеют разные размеры?

**Мои ответы:**

1. `Dataset` определяет количество samples и способ получения одного sample по индексу. `DataLoader` выбирает индексы, при необходимости перемешивает их, получает samples из Dataset и объединяет их в batches.
2. Sample — один объект. Batch — группа объектов, обрабатываемая вместе. Iteration — обработка одного batch и обычно один optimizer step. Epoch — один полный проход по обучающему Dataset.
3. Если размер Dataset не делится на `batch_size` без остатка и `drop_last=False`, последний batch содержит оставшиеся объекты и поэтому имеет меньший размер.
4. `drop_last=True` отбрасывает последний неполный batch. Это полезно, если алгоритму необходим постоянный batch size или очень маленький batch мешает BatchNorm; недостаток — часть объектов не используется в этой эпохе.
5. Train loader обычно перемешивают, чтобы batches не сохраняли потенциально смещённый порядок данных. На validation параметры не обновляются, а стабильный порядок повышает воспроизводимость и облегчает анализ отдельных предсказаний.
6. Если loss каждого batch уже усреднён, нужно вычислить взвешенное среднее: `epoch_loss = sum(batch_loss * batch_size) / total_objects`. Обычное среднее batch losses неверно, когда batches имеют разные размеры.

In [8]:
def train_experiment(
    batch_size,
    *,
    shuffle=True,
    drop_last=False,
    num_epochs=100,
    learning_rate=0.02,
):
    # Одинаковая инициализация для честного сравнения.
    torch.manual_seed(42)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
    )

    model = LinearRegression()
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=learning_rate,
    )

    global_step = 0
    seen_objects_total = 0
    last_epoch_loss = None

    model.train()

    for epoch in range(num_epochs):
        epoch_loss_sum = 0.0
        seen_objects = 0

        for batch_x, batch_y in loader:
            optimizer.zero_grad()

            prediction = model(batch_x)
            loss = loss_fn(prediction, batch_y)

            loss.backward()
            optimizer.step()

            current_batch_size = batch_x.shape[0]
            epoch_loss_sum += loss.item() * current_batch_size
            seen_objects += current_batch_size

            global_step += 1
            seen_objects_total += current_batch_size

        last_epoch_loss = epoch_loss_sum / seen_objects

    return {
        "batch_size": batch_size,
        "batches_per_epoch": len(loader),
        "global_steps": global_step,
        "seen_objects_total": seen_objects_total,
        "weight": model.linear.weight.item(),
        "bias": model.linear.bias.item(),
        "loss": last_epoch_loss,
    }

In [9]:
results = [
    train_experiment(batch_size=1),
    train_experiment(batch_size=4),
    train_experiment(batch_size=10),
]

for result in results:
    print(result)

{'batch_size': 1, 'batches_per_epoch': 10, 'global_steps': 1000, 'seen_objects_total': 1000, 'weight': 1.9999998807907104, 'bias': 1.0000011920928955, 'loss': 2.0351365037640788e-11}
{'batch_size': 4, 'batches_per_epoch': 3, 'global_steps': 300, 'seen_objects_total': 1000, 'weight': 1.9994574785232544, 'bias': 1.002592921257019, 'loss': 2.2598735995416062e-06}
{'batch_size': 10, 'batches_per_epoch': 1, 'global_steps': 100, 'seen_objects_total': 1000, 'weight': 1.9986498355865479, 'bias': 1.0084662437438965, 'loss': 2.1230402126093395e-05}


In [10]:
keep_last = train_experiment(
    batch_size=4,
    drop_last=False,
)

drop_last = train_experiment(
    batch_size=4,
    drop_last=True,
)

print(keep_last)
print(drop_last)

{'batch_size': 4, 'batches_per_epoch': 3, 'global_steps': 300, 'seen_objects_total': 1000, 'weight': 1.9994574785232544, 'bias': 1.002592921257019, 'loss': 2.2598735995416062e-06}
{'batch_size': 4, 'batches_per_epoch': 2, 'global_steps': 200, 'seen_objects_total': 800, 'weight': 1.99867844581604, 'bias': 1.007398009300232, 'loss': 1.827999267334235e-05}


In [11]:
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
)

batch_losses = []

with torch.inference_mode():
    for batch_x, batch_y in loader:
        prediction = model(batch_x)
        loss = loss_fn(prediction, batch_y)

        batch_losses.append(
            (loss.item(), batch_x.shape[0])
        )

print(batch_losses)

[(0.00023939467791933566, 4), (7.55143555579707e-05, 4), (0.0004932512529194355, 2)]


In [12]:
naive_mean = sum(
    loss for loss, batch_size in batch_losses
) / len(batch_losses)

In [13]:
weighted_mean = sum(
    loss * batch_size
    for loss, batch_size in batch_losses
) / sum(
    batch_size
    for loss, batch_size in batch_losses
)

In [14]:
with torch.inference_mode():
    direct_loss = loss_fn(model(x), y).item()

print("naive:", naive_mean)
print("weighted:", weighted_mean)
print("direct:", direct_loss)

naive: 0.00026938676213224727
weighted: 0.00022461386397480964
direct: 0.0002246138610644266
